# 02 - Data Cleaning & SQL-ready Preparation

**Goal:** turn the raw files into reproducible, documented, analysis-ready tables without modifying the originals.

**Principles**

* Raw files are immutable.
* Every imputation is explicit and flagged.
* A LEFT JOIN must never change the row count.
* No statistic used for imputation is computed from future validation or test data.

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110
print("Project root:", PROJECT_ROOT)

Project root: rossmann-sales-forecasting


## 1. Cleaning the store dimension

`CompetitionDistance` has 3 missing values; they are replaced by the median and flagged. Competition start dates are kept unknown rather than invented. `Promo2` dates are structurally missing for non-participating stores.

In [2]:
from src.data_preparation import clean_store, load_raw_data
from src.config import load_config

cfg = load_config()
raw = load_raw_data(cfg)
store_raw = raw["store"]
store = clean_store(store_raw)

comparison = pd.DataFrame({
    "raw_missing": store_raw.isna().sum(),
    "clean_missing": store.isna().sum().reindex(store_raw.columns, fill_value=0),
})
display(comparison)
display(store[["Store", "StoreType", "Assortment", "CompetitionDistance", "CompetitionDistanceMissing",
               "CompetitionOpenSince", "HasCompetitionStart", "Promo2", "Promo2Start", "Promo2Active" if "Promo2Active" in store else "Promo2MonthsCsv"]].head(5))

,raw_missing,clean_missing
Store,0,0
StoreType,0,0
Assortment,0,0
CompetitionDistance,3,0
CompetitionOpenSinceMonth,354,354
CompetitionOpenSinceYear,354,354
Promo2,0,0
Promo2SinceWeek,544,544
Promo2SinceYear,544,544
PromoInterval,544,0


,Store,StoreType,Assortment,CompetitionDistance,CompetitionDistanceMissing,CompetitionOpenSince,HasCompetitionStart,Promo2,Promo2Start,Promo2MonthsCsv
0,1,c,a,1270.0,0,2008-09-01,1,0,NaT,
1,2,a,a,570.0,0,2007-11-01,1,1,2010-03-29,"1,4,7,10"
2,3,a,a,14130.0,0,2006-12-01,1,1,2011-04-04,"1,4,7,10"
3,4,c,c,620.0,0,2009-09-01,1,0,NaT,
4,5,a,a,29910.0,0,2015-04-01,1,0,NaT,


## 2. Cleaning the transaction tables

* Dates parsed to daily timestamps.
* `StateHoliday` normalised to `0/a/b/c` plus three binary flags.
* Missing `Open` (11 test rows) imputed from the store's modal state and flagged.
* `SalesZeroWhileOpen` marks the 54 anomalies.
* Closed days are **kept** in the training table: they are needed to build correct lag and rolling features (a Sunday closure is real information about the store's weekly pattern).

In [3]:
from src.data_preparation import clean_sales

train_raw = raw["train"]
test_raw = raw["test"]
train = clean_sales(train_raw, is_train=True)
test = clean_sales(test_raw, is_train=False)

print(f"Train rows: {len(train_raw):,} -> {len(train):,}")
print(f"Test rows : {len(test_raw):,} -> {len(test):,}")
print("Open values imputed in test:", int(test.OpenImputed.sum()))
print("Open-but-zero sales in train:", int(train.SalesZeroWhileOpen.sum()))
display(train[["Store", "Date", "Sales", "Customers", "Open", "Promo", "StateHoliday",
               "StateHoliday_a", "IsStateHoliday", "SalesZeroWhileOpen"]].head(5))

Imputed 11 missing Open values in the test split


Train rows: 1,017,209 -> 1,017,209
Test rows : 41,088 -> 41,088
Open values imputed in test: 11
Open-but-zero sales in train: 54


,Store,Date,Sales,Customers,Open,Promo,StateHoliday,StateHoliday_a,IsStateHoliday,SalesZeroWhileOpen
0,1,2015-07-31,5263,555,1,1,0,0,0,0
1,2,2015-07-31,6064,625,1,1,0,0,0,0
2,3,2015-07-31,8314,821,1,1,0,0,0,0
3,4,2015-07-31,13995,1498,1,1,0,0,0,0
4,5,2015-07-31,4822,559,1,1,0,0,0,0


## 3. Joining store attributes

The join is validated as `many_to_one`; any unmatched store would raise an error instead of silently producing nulls.

In [4]:
from src.data_preparation import merge_datasets

train_clean = merge_datasets(train, store)
test_clean = merge_datasets(test, store)
print("Train shape after join:", train_clean.shape)
print("Test shape after join :", test_clean.shape)
print("Any unmatched stores  :", train_clean.StoreType.isna().any() or test_clean.StoreType.isna().any())
display(train_clean[["Store", "Date", "Sales", "StoreType", "Assortment", "CompetitionDistance",
                     "Promo2", "Promo2Start"]].head(5))

Train shape after join: (1017209, 33)
Test shape after join : (41088, 30)
Any unmatched stores  : False


,Store,Date,Sales,StoreType,Assortment,CompetitionDistance,Promo2,Promo2Start
0,1,2015-07-31,5263,c,a,1270.0,0,NaT
1,2,2015-07-31,6064,a,a,570.0,1,2010-03-29
2,3,2015-07-31,8314,a,a,14130.0,1,2011-04-04
3,4,2015-07-31,13995,c,c,620.0,0,NaT
4,5,2015-07-31,4822,a,a,29910.0,0,NaT


## 4. Automated quality checks

The pipeline writes the result of every check to `outputs/metrics/data_quality.json`. The table below is the acceptance gate for the cleaning stage.

In [5]:
from src.data_preparation import prepare_data

checks = prepare_data(cfg, write=False)
display(pd.DataFrame(checks["checks"]))
print("All checks passed:", checks["tests_passed"])

Imputed 11 missing Open values in the test split


,check,passed,detail
0,row_count_preserved_by_join,True,LEFT JOIN must not duplicate rows
1,unique_store_date_train,True,one row per store-day
2,unique_store_date_test,True,one row per store-day
3,no_missing_store_type,True,all stores matched store.csv
4,non_negative_sales,True,Sales >= 0
5,date_ranges_contiguous,True,test starts the day after train
6,all_test_stores_in_train,True,cold-start stores require separate handling
7,submission_rows_match_test,True,"official test split contains 41,088 store-day ..."


All checks passed: True


## 5. SQL model

The relational model is intentionally simple: one fact table (`train_sales`) and two dimensions (`stores`, `test_sales`). It supports JOIN, GROUP BY, CTE and window-function queries without duplicating store attributes on every row.

The database is built by:

```bash
python sql/build_database.py
```

It creates `data/processed/rossmann.sqlite`, runs all 13 named queries from `sql/analytical_queries.sql` and exports each result to `outputs/metrics/sql/`. The readable summary is `outputs/metrics/sql_results.md`.

**Why a star schema?** Store attributes are static; keeping them in a dimension avoids update anomalies and makes the analytical queries shorter and less error-prone. The fact table carries only the transactional grain (one row per store-day).